# Autonomous electric vehicle — ThunderGraph model demo

This notebook sketches a **Level-4-capable autonomous electric vehicle (AEV)** using **thundergraph-model** with **two parallel hierarchies**: **nested subsystems** (a `System` may contain child `System`s and leaf parts) and **nested requirements** (`Requirement` + `model.requirement_package`, dot notation on `RequirementRef`).

**Patterns shown:** `parameter` / `attribute(expr=…)` / `constraint(expr=…)` on **parts** and on **nested composable requirement packages** (`Requirement.define`): package-level slots live under **`cm.requirements.<pkg>.*`** (same authoring surface as parts). **Leaf** **`model.requirement(...)`** carries the **statement text** and **`references`**; **executable checks** are **`model.constraint`** on the package (not `requirement_input` / `requirement_accept_expr` in this demo). **`model.allocate`** still links each leaf requirement to its **realizing** part for traceability. For **Atlas-400F**-style **leaf** acceptance with **`requirement_input`** + **`requirement_accept_expr`**, see `cargo_jet_program.ipynb`.

**Evaluation note:** Package **parameters** (e.g. `requirements.propulsion.shaft_power`) must be **bound in `evaluate(inputs=…)`** like part parameters. This notebook mirrors the **same** physical quantities onto those slots (traction power from torque×speed; motion/planner scalars duplicated from the parts) so one run closes both part and requirement-package graphs.

**Run:** From the `thundergraph-model` directory: `uv sync --group dev` (or `--all-groups`), then run cells top to bottom, or execute headless with `uv run --group dev python -m jupyter nbconvert --execute --inplace notebooks/autonomous_electric_vehicle.ipynb`.


## 1. Requirements vs structure (parallel trees)

| Concern | Requirement (nested under `requirements.*`) | Realized on (allocate target) |
|--------|-----------------------------------------------|------------------------------|
| **R-TRACTION** | `requirements.propulsion.traction` | `vehicle_propulsion.powertrain` |
| **R-AUTO** | `requirements.motion.autonomy_gate` | `motion` |
| **R-FLOW** | `requirements.perception.fusion_path` | `autonomy.planner` (perception → planner path) |
| **R-SAFE** | `requirements.safety.fail_safe` (text + allocate; no `expr`) | `safety` |

Formal requirements live under **`model.requirement_package("requirements", …)`** on **`AutonomousVehicle`**, not inside leaf part `define()` blocks. Subsystems are **`System`** types composed under the vehicle (`vehicle_propulsion`, `autonomy`) so **structure mirrors requirement grouping**.


## 2. Architecture (nested systems)

- **`VehiclePropulsion` (System)** — child **`powertrain`** (`PowertrainPart`): derived **`shaft_power`**, constraint, traction FSM, SOC binding.
- **`motion`** — top-level `MotionPart` (manual ↔ autonomous guard).
- **`DriveByWireSubsystem` (System)** — **`perception`** + **`planner`** (ports + planner constraint); the **structural** `Observation` link is **`model.connect`** on the **vehicle** so it appears in **`ConfiguredModel.connections`** (R-FLOW).
- **`safety`** — `SafetySupervisorPart` sequences and fork/join.

**Cross-cutting:** **`model.connect(aut.perception.scene_out → aut.planner.fusion_in)`** is declared on **`AutonomousVehicle`** so **`ConfiguredModel.connections`** includes the item-flow binding (nested-type edges are not auto-hoisted). Scenario **`expected_interaction_order`** uses **dotted part paths** from the vehicle root (e.g. `vehicle_propulsion.powertrain`, `autonomy.planner`).


In [1]:
# nbconvert often sets cwd to `notebooks/`; cwd may also be monorepo root — find `tg_model` reliably.
from __future__ import annotations


from unitflow import Quantity
from unitflow.catalogs.si import N, m, rad, s
from unitflow.core.units import Unit

# String metadata like unit="1" breaks unitflow symbols in relational expr; use dimensionless Unit.
DIMLESS = Unit.dimensionless()

from tg_model.model.elements import Part, Requirement, System
from tg_model.execution.configured_model import instantiate
from tg_model.execution.evaluator import Evaluator
from tg_model.execution.graph_compiler import compile_graph
from tg_model.execution.requirements import summarize_requirement_satisfaction
from tg_model.execution.run_context import RunContext
from tg_model.execution.validation import validate_graph
from tg_model.execution.instances import PartInstance
from tg_model.execution.behavior import (
    BehaviorTrace,
    DecisionDispatchOutcome,
    DispatchOutcome,
    behavior_authoring_projection,
    behavior_trace_to_records,
    dispatch_decision,
    dispatch_event,
    dispatch_fork_join,
    dispatch_sequence,
    emit_item,
    validate_scenario_trace,
)


In [2]:
def _reset_aev_types() -> None:
    for cls in (
        PowertrainPart,
        MotionPart,
        PerceptionStackPart,
        PlannerPart,
        SafetySupervisorPart,
        VehiclePropulsion,
        DriveByWireSubsystem,
        AEVPropulsionReqs,
        AEVMotionReqs,
        AEVPerceptionReqs,
        AEVSafetyReqs,
        AEVRequirementsRoot,
        AutonomousVehicle,
    ):
        cls._reset_compilation()


class PowertrainPart(Part):
    """Battery + inverter + traction: derived shaft power + part constraints; discrete traction FSM."""

    @classmethod
    def define(cls, model):  # type: ignore[override]
        # Requirements are NOT declared on parts — only parameters, attributes, constraints, behavior.
        cmd_torque = model.parameter("cmd_torque", unit=N * m)
        shaft_speed = model.parameter("shaft_speed", unit=rad / s)
        shaft_power = model.attribute(
            "shaft_power",
            unit=N * m / s,
            expr=cmd_torque * shaft_speed,
        )
        c_shaft_nn = model.constraint(
            "shaft_power_non_negative",
            expr=shaft_power >= Quantity(0, N * m / s),
        )
        cite_traction = model.citation(
            "cite_traction_power",
            title="Illustrative EV traction power envelope (fake)",
            standard_id="DEMO-SAE-EV-001",
        )
        model.references(c_shaft_nn, cite_traction)
        soc = model.parameter("soc", unit="1")
        st = model.state("standby", initial=True)
        pr = model.state("propulsion")
        flt = model.state("fault")
        ev_enable = model.event("enable_traction")
        ev_fault = model.event("fault_event")
        ev_reset = model.event("reset")

        def apply_torque(c: RunContext, p: PartInstance) -> None:
            c.bind_input(p.soc.stable_id, 0.92)

        model.action("apply_torque", effect=apply_torque)
        model.transition(st, pr, on=ev_enable, effect="apply_torque")
        model.transition(pr, flt, on=ev_fault)
        model.transition(st, flt, on=ev_fault)
        model.transition(flt, st, on=ev_reset)


class MotionPart(Part):
    """Operating mode: manual vs autonomous with a guard on engagement."""

    @classmethod
    def define(cls, model):  # type: ignore[override]
        auton_cmd_ok = model.parameter("auton_cmd_ok", unit=DIMLESS)
        min_supervisor_score = model.parameter("min_supervisor_score", unit=DIMLESS)
        manual = model.state("manual", initial=True)
        auto = model.state("autonomous")
        engage = model.event("engage_auto")
        disengage = model.event("disengage")
        g = model.guard(
            "auton_ok",
            predicate=lambda c, p: c.get_value(p.auton_cmd_ok.stable_id) >= 1.0,
        )
        model.transition(manual, auto, on=engage, guard=g)
        model.transition(auto, manual, on=disengage)


class PerceptionStackPart(Part):
    """Publishes fused scene observations on a structural port."""

    @classmethod
    def define(cls, model):  # type: ignore[override]
        model.port("scene_out", direction="out")
        model.item_kind("Observation")


class PlannerPart(Part):
    """Consumes observations at a port; transition on item-carrying event."""

    @classmethod
    def define(cls, model):  # type: ignore[override]
        model.port("fusion_in", direction="in")
        fused_stamp = model.parameter("fused_stamp", unit=DIMLESS)
        min_valid_stamp = model.parameter("min_valid_stamp", unit=DIMLESS)
        c_fusion = model.constraint(
            "fusion_stamp_ge_threshold",
            expr=fused_stamp >= min_valid_stamp,
        )
        cite_fusion = model.citation(
            "cite_fusion_latency",
            title="Illustrative fusion pipeline timing basis (fake)",
            doi="10.0000/demo.fusion.042",
        )
        model.references(c_fusion, cite_fusion)

        def ingest(c: RunContext, p: PartInstance) -> None:
            pl = c.peek_item_payload(p.path_string, "Observation")
            if pl is not None:
                c.bind_input(p.fused_stamp.stable_id, float(pl))

        model.action("ingest", effect=ingest)
        obs = model.event("Observation")
        idle = model.state("idle", initial=True)
        busy = model.state("busy")
        model.transition(idle, busy, on=obs, effect="ingest")


class SafetySupervisorPart(Part):
    """Redundant checks: linear preflight then fork/join style dual path (v0 serial)."""

    @classmethod
    def define(cls, model):  # type: ignore[override]
        model.action("check_comms", effect=lambda c, p: None)
        model.action("check_estop", effect=lambda c, p: None)
        model.action("check_sensor_a", effect=lambda c, p: None)
        model.action("check_sensor_b", effect=lambda c, p: None)
        model.action("mark_ready", effect=lambda c, p: None)
        model.sequence("preflight", steps=["check_comms", "check_estop"])
        model.fork_join(
            "dual_check",
            branches=[["check_sensor_a"], ["check_sensor_b"]],
            then_action="mark_ready",
        )


class VehiclePropulsion(System):
    """Nested system: composed traction stack (maps to R-TRACTION allocation layer)."""

    @classmethod
    def define(cls, model):  # type: ignore[override]
        model.part("powertrain", PowertrainPart)


class DriveByWireSubsystem(System):
    """Nested system: perception + planner parts (R-FLOW path).

    The structural ``connect`` is declared on :class:`AutonomousVehicle` so it appears in the
    root compiled edges (``ConfiguredModel.connections``). Nested-type ``connect`` edges are
    not automatically hoisted today — same pattern as a flat model, but parts live under
    ``autonomy.*``.
    """

    @classmethod
    def define(cls, model):  # type: ignore[override]
        model.part("perception", PerceptionStackPart)
        model.part("planner", PlannerPart)


class AEVPropulsionReqs(Requirement):
    @classmethod
    def define(cls, model):  # type: ignore[override]
        r = model.requirement(
            "traction",
            "Traction mechanical power shall be non-negative under command.",
        )
        shaft_power = model.parameter("shaft_power", unit=N * m / s)
        model.constraint(
            "traction_non_negative",
            expr=shaft_power >= Quantity(0, N * m / s),
        )
        c = model.citation(
            "cite_traction_req",
            title="Illustrative EV traction power envelope (fake)",
            standard_id="DEMO-SAE-EV-001",
        )
        model.references(r, c)


class AEVMotionReqs(Requirement):
    @classmethod
    def define(cls, model):  # type: ignore[override]
        r = model.requirement(
            "autonomy_gate",
            "Autonomy engagement shall require supervisor readiness (cmd ≥ threshold).",
        )
        auton_cmd_ok = model.parameter("auton_cmd_ok", unit=DIMLESS)
        min_supervisor_score = model.parameter("min_supervisor_score", unit=DIMLESS)
        autonomy_margin = model.attribute(
            "autonomy_margin",
            unit=DIMLESS,
            expr=auton_cmd_ok - min_supervisor_score,
        )
        model.constraint(
            "autonomy_gate_ok",
            expr=autonomy_margin >= Quantity(0, DIMLESS),
        )
        c = model.citation(
            "cite_motion_req",
            title="Illustrative ODD / engagement policy note (fake)",
            standard_id="DEMO-SAE-J3016",
        )
        model.references(r, c)


class AEVPerceptionReqs(Requirement):
    @classmethod
    def define(cls, model):  # type: ignore[override]
        r = model.requirement(
            "fusion_path",
            "Fusion stamp shall meet minimum threshold (perception → planner path).",
        )
        fused_stamp = model.parameter("fused_stamp", unit=DIMLESS)
        min_valid_stamp = model.parameter("min_valid_stamp", unit=DIMLESS)
        fusion_headroom = model.attribute(
            "fusion_headroom",
            unit=DIMLESS,
            expr=fused_stamp - min_valid_stamp,
        )
        model.constraint(
            "fusion_stamp_ok",
            expr=fusion_headroom >= Quantity(0, DIMLESS),
        )
        c = model.citation(
            "cite_perception_req",
            title="Illustrative fusion pipeline timing basis (fake)",
            doi="10.0000/demo.fusion.042",
        )
        model.references(r, c)


class AEVSafetyReqs(Requirement):
    @classmethod
    def define(cls, model):  # type: ignore[override]
        r = model.requirement(
            "fail_safe",
            "Vehicle shall maintain a safe state when supervisory checks fail.",
        )
        c = model.citation(
            "cite_func_safety",
            title="Illustrative functional safety lifecycle (fake)",
            standard_id="DEMO-ISO-26262",
        )
        model.references(r, c)


class AEVRequirementsRoot(Requirement):
    """Nested requirement tree: dot notation e.g. ``req_root.propulsion.traction``."""

    @classmethod
    def define(cls, model):  # type: ignore[override]
        model.requirement_package("propulsion", AEVPropulsionReqs)
        model.requirement_package("motion", AEVMotionReqs)
        model.requirement_package("perception", AEVPerceptionReqs)
        model.requirement_package("safety", AEVSafetyReqs)


class AutonomousVehicle(System):
    """Root system: nested ``requirements`` package + nested ``System`` children."""

    @classmethod
    def define(cls, model):  # type: ignore[override]
        vp = model.part("vehicle_propulsion", VehiclePropulsion)
        motion = model.part("motion", MotionPart)
        aut = model.part("autonomy", DriveByWireSubsystem)
        safe = model.part("safety", SafetySupervisorPart)

        req_root = model.requirement_package("requirements", AEVRequirementsRoot)

        cite_j3016 = model.citation(
            "cite_j3016",
            title="Illustrative autonomy taxonomy note (fake)",
            standard_id="DEMO-SAE-J3016",
        )
        model.references(req_root.motion.autonomy_gate, cite_j3016)
        model.references(req_root.perception.fusion_path, cite_j3016)

        model.allocate(req_root.propulsion.traction, vp.powertrain)
        model.allocate(req_root.motion.autonomy_gate, motion)
        model.allocate(req_root.perception.fusion_path, aut.planner)
        model.allocate(req_root.safety.fail_safe, safe)

        model.connect(
            source=aut.perception.scene_out,
            target=aut.planner.fusion_in,
            carrying="Observation",
        )
        model.scenario(
            "sensor_to_plan",
            expected_event_order=[],
            expected_interaction_order=[
                ("vehicle_propulsion.powertrain", "enable_traction"),
                ("motion", "engage_auto"),
                ("autonomy.planner", "Observation"),
            ],
            expected_item_kind_order=["Observation"],
        )


_reset_aev_types()


## 3. Derived attributes, part constraints, and requirement packages

**Leaf parts** declare parameters, attributes (`expr=`), constraints, ports, and behavior — **not** formal requirement statements.

**Composable requirement packages** (`AEVPropulsionReqs`, …) use the **same** value vocabulary as parts: **`model.parameter`**, **`model.attribute`**, **`model.constraint`**. **Leaf** **`model.requirement`** nodes hold **text** and **`model.references`**; **`model.allocate`** ties each leaf requirement to the **part** that realizes it (no `inputs=` here because checks read **package** parameters). **`summarize_requirement_satisfaction`** only lists **requirement-tagged** acceptance rows (`reqcheck:…` from **`requirement_accept_expr`**); this demo’s checks are **package constraints** and appear in the general **constraint** list with names like **`…requirements.propulsion.traction_non_negative`**.

**Phase 8:** Citations and **`model.references`** are declared **inside** each subdomain **requirement package** where possible; the vehicle adds extra **`cite_j3016`** edges to motion and perception requirements.

**`requirements.safety.fail_safe`** is **text + allocate only** (no package `constraint`), so it does not add a check node.

In [3]:
_reset_aev_types()
cm = instantiate(AutonomousVehicle)
graph, handlers = compile_graph(cm)
v = validate_graph(graph)
assert v.passed, v.failures

evalr = Evaluator(graph, compute_handlers=handlers)
ctxv = RunContext()
_tq = Quantity(800, N * m)
_spd = Quantity(120, rad / s)
_shaft = _tq * _spd
_acmd = Quantity(1.0, DIMLESS)
_mins = Quantity(1.0, DIMLESS)
_fused = Quantity(20_250_324.0, DIMLESS)
_fmin = Quantity(1.0, DIMLESS)
result = evalr.evaluate(
    ctxv,
    inputs={
        cm.vehicle_propulsion.powertrain.cmd_torque.stable_id: _tq,
        cm.vehicle_propulsion.powertrain.shaft_speed.stable_id: _spd,
        cm.vehicle_propulsion.powertrain.soc.stable_id: 0.88,
        cm.motion.auton_cmd_ok.stable_id: _acmd,
        cm.motion.min_supervisor_score.stable_id: _mins,
        cm.autonomy.planner.fused_stamp.stable_id: _fused,
        cm.autonomy.planner.min_valid_stamp.stable_id: _fmin,
        cm.requirements.propulsion.shaft_power.stable_id: _shaft,
        cm.requirements.motion.auton_cmd_ok.stable_id: _acmd,
        cm.requirements.motion.min_supervisor_score.stable_id: _mins,
        cm.requirements.perception.fused_stamp.stable_id: _fused,
        cm.requirements.perception.min_valid_stamp.stable_id: _fmin,
    },
)

sp = ctxv.get_value(cm.vehicle_propulsion.powertrain.shaft_power.stable_id)
print("shaft_power (cmd_torque × shaft_speed):", sp)


def _ascii_table(headers: list[str], rows: list[list[str]]) -> str:
    """Simple box-drawing table (stdlib only)."""
    cols = len(headers)
    cells: list[list[str]] = [[str(h) for h in headers]]
    for r in rows:
        row = [str(c) for c in r]
        if len(row) < cols:
            row.extend([""] * (cols - len(row)))
        cells.append(row[:cols])
    widths = [0] * cols
    for r in cells:
        for i, c in enumerate(r):
            widths[i] = max(widths[i], len(c))
    sep = "+" + "+".join("-" * (w + 2) for w in widths) + "+"

    def line(vals: list[str]) -> str:
        return "|" + "|".join(f" {vals[i].ljust(widths[i])} " for i in range(cols)) + "|"

    out = [sep, line(cells[0]), sep]
    for r in cells[1:]:
        out.append(line(r))
    out.append(sep)
    return "\n".join(out)


def _aev_constraint_rows(result):
    root = "AutonomousVehicle."
    rows = []
    for cr in result.constraint_results:
        full = cr.name
        rel = full[len(root) :] if full.startswith(root) else full
        segs = rel.split(".")
        cname = segs[-1]
        if rel.startswith("requirements."):
            layer = "req package"
            scope = ".".join(segs[:2]) if len(segs) >= 2 else rel
        else:
            layer = "part"
            scope = ".".join(segs[:-1]) if len(segs) > 1 else rel
        if "requirements.propulsion" in full:
            note = "R-TRACTION package (leaf: traction)"
        elif "requirements.motion" in full:
            note = "R-AUTO package (leaf: autonomy_gate)"
        elif "requirements.perception" in full:
            note = "R-FLOW package (leaf: fusion_path)"
        elif "powertrain" in full:
            note = "mirrors requirements.propulsion.shaft_power"
        elif "planner" in full:
            note = "planner stamp guard (+ perception package)"
        else:
            note = "—"
        status = "pass" if cr.passed else "FAIL"
        rows.append([layer, scope, cname, status, note, full])
    rows.sort(key=lambda r: (0 if r[0] == "req package" else 1, r[1], r[2]))
    return rows


summary = summarize_requirement_satisfaction(result)
rows = _aev_constraint_rows(result)
n = len(rows)
n_pass = sum(1 for r in rows if r[3] == "pass")
head = f"Constraint satisfaction ({n_pass}/{n} passed)"
print(head)
print(
    _ascii_table(
        ["layer", "scope", "constraint", "status", "note", "full graph id"],
        [r[:6] for r in rows],
    )
)
if summary.check_count == 0:
    print(
        "Leaf reqcheck rows: none (use requirement_accept_expr e.g. cargo_jet_program.ipynb); "
        "checks above are package/part constraints."
    )
else:
    print("Tagged requirement acceptance (reqcheck)")
    print(
        _ascii_table(
            ["requirement", "allocation", "check", "status", "evidence"],
            [
                [
                    s.requirement_path,
                    s.allocation_target_path,
                    s.check_name,
                    "pass" if s.passed else "FAIL",
                    s.evidence or "—",
                ]
                for s in summary.results
            ],
        )
    )

print("Provenance (model.references → citation):", len(cm.references))
for rb in cm.references:
    print(" ", rb)

shaft_power (cmd_torque × shaft_speed): 96000 m^2 * kg / s^3
Constraint satisfaction (5/5 passed)
+-------------+-------------------------------+---------------------------+--------+---------------------------------------------+--------------------------------------------------------------------------+
| layer       | scope                         | constraint                | status | note                                        | full graph id                                                            |
+-------------+-------------------------------+---------------------------+--------+---------------------------------------------+--------------------------------------------------------------------------+
| req package | requirements.motion           | autonomy_gate_ok          | pass   | R-AUTO package (leaf: autonomy_gate)        | AutonomousVehicle.requirements.motion.autonomy_gate_ok                   |
| req package | requirements.perception       | fusion_stamp_ok           | pa

## 4. Compile and inspect structure

The root compiled artifact includes **nodes** (parts, nested **composable requirement packages** under **`requirements`** — each package node still has internal compiled **`kind`** **`"requirement_block"`** for artifact compatibility), **edges** (**connect**, **allocate**, **references**), and **behavior** transition summaries. Open **`DriveByWireSubsystem.compile()`** separately to see **`perception`** / **`planner`** without root-level **connect** (that edge lives on the vehicle).


In [4]:
_reset_aev_types()
av = AutonomousVehicle.compile()
print("Owner:", av["owner"])
print("Top-level parts:", [k for k, v in av["nodes"].items() if v["kind"] == "part"])
# Filter by internal artifact kind (unchanged string; public API is requirement_package / Requirement).
req_pkg_nodes = [k for k, v in av["nodes"].items() if v["kind"] == "requirement_block"]
print("Requirement packages at root (internal kind requirement_block):", req_pkg_nodes)
print("Edges:", len(av["edges"]))
for e in av["edges"]:
    print(" ", e["kind"], e.get("carrying") or "")
print("Behavior transitions (sample):", av["behavior_transitions"][:3], "...")


Owner: __main__.AutonomousVehicle
Top-level parts: ['vehicle_propulsion', 'motion', 'autonomy', 'safety']
Requirement packages at root (internal kind requirement_block): ['requirements']
Edges: 7
  references 
  references 
  allocate 
  allocate 
  allocate 
  allocate 
  connect Observation
Behavior transitions (sample): [] ...


## 5. Authoring projection (diagram / export hook)

`behavior_authoring_projection` returns a JSON-friendly summary of discrete behavior nodes per **part type**.


In [5]:
_reset_aev_types()
proj = behavior_authoring_projection(SafetySupervisorPart)
print({k: proj[k] for k in ("sequences", "fork_joins", "actions")})


{'sequences': ['preflight'], 'fork_joins': ['dual_check'], 'actions': ['check_comms', 'check_estop', 'check_sensor_a', 'check_sensor_b', 'mark_ready']}


## 6. Instantiate and run a scripted mission

Same **`RunContext`** holds parameters and discrete state for all parts. Order below: traction → engage autopilot (guard) → safety **sequence** + **fork_join** → **emit_item** to planner. Bind **`min_supervisor_score`** so the motion guard matches the acceptance policy used in §3.


In [6]:
_reset_aev_types()
cm = instantiate(AutonomousVehicle)
ctx = RunContext()
ctx.bind_input(cm.motion.auton_cmd_ok.stable_id, 1.0)
ctx.bind_input(cm.motion.min_supervisor_score.stable_id, 1.0)
ctx.bind_input(cm.vehicle_propulsion.powertrain.soc.stable_id, 0.9)

tr = BehaviorTrace()

r1 = dispatch_event(ctx, cm.vehicle_propulsion.powertrain, "enable_traction", trace=tr)
r2 = dispatch_event(ctx, cm.motion, "engage_auto", trace=tr)
dispatch_sequence(ctx, cm.safety, "preflight", trace=tr)
dispatch_fork_join(ctx, cm.safety, "dual_check", trace=tr)
out = emit_item(ctx, cm, cm.autonomy.perception.scene_out, "Observation", 20250324.0, trace=tr)

print("Traction dispatch:", r1.outcome)
print("Autopilot dispatch:", r2.outcome)
print("emit_item branch results:", [x.outcome for x in out])
print("Planner fused_stamp:", ctx.get_value(cm.autonomy.planner.fused_stamp.stable_id))
print("Powertrain soc:", ctx.get_value(cm.vehicle_propulsion.powertrain.soc.stable_id))
print("Motion mode:", ctx.get_active_behavior_state(cm.motion.path_string))


Traction dispatch: fired
Autopilot dispatch: fired
emit_item branch results: [<DispatchOutcome.FIRED: 'fired'>]
Planner fused_stamp: 20250324.0
Powertrain soc: 0.92
Motion mode: autonomous


## 7. Optional: decision + merge (single-part pattern)

Not used on the vehicle root above; **`MotionPart`** could use a **`decision`** node instead of a guarded transition for the same intent. Here is a minimal **temperature routing** example on a throwaway part (shows `DecisionDispatchResult`).


In [7]:
class ThermalRouter(Part):
    @classmethod
    def define(cls, model):  # type: ignore[override]
        model.parameter("temp", unit="1")
        hot = model.guard("hot", predicate=lambda c, p: c.get_value(p.temp.stable_id) > 40.0)
        model.action("cool", effect=lambda c, p: c.bind_input(p.temp.stable_id, 25.0))
        model.action("hold", effect=lambda c, p: None)
        model.decision("route", branches=[(hot, "cool")], default_action="hold")


ThermalRouter._reset_compilation()
ctx2 = RunContext()
cm_tr = instantiate(ThermalRouter)
ctx2.bind_input(cm_tr.temp.stable_id, 55.0)
tr2 = BehaviorTrace()
r = dispatch_decision(ctx2, cm_tr, "route", trace=tr2)
print(r.outcome, r.chosen_action, bool(r))
assert r.outcome == DecisionDispatchOutcome.ACTION_RAN


action_ran cool True


## 8. Scenario validation (partial contract)

`validate_scenario_trace` bundles checks: **global transition order** for `expected_interaction_order` (state-machine steps only), **item kind order**, etc. Re-run after changing the trace.


In [8]:
_reset_aev_types()

cm = instantiate(AutonomousVehicle)
ctx = RunContext()
ctx.bind_input(cm.motion.auton_cmd_ok.stable_id, 1.0)
ctx.bind_input(cm.motion.min_supervisor_score.stable_id, 1.0)
ctx.bind_input(cm.vehicle_propulsion.powertrain.soc.stable_id, 0.9)
tr = BehaviorTrace()
dispatch_event(ctx, cm.vehicle_propulsion.powertrain, "enable_traction", trace=tr)
dispatch_event(ctx, cm.motion, "engage_auto", trace=tr)
dispatch_sequence(ctx, cm.safety, "preflight", trace=tr)
dispatch_fork_join(ctx, cm.safety, "dual_check", trace=tr)
emit_item(ctx, cm, cm.autonomy.perception.scene_out, "Observation", 20250324.0, trace=tr)

ok, errors = validate_scenario_trace(
    definition_type=AutonomousVehicle,
    scenario_name="sensor_to_plan",
    part_path=cm.path_string,
    trace=tr,
    root=cm.root,
)
print("Scenario OK:", ok)
print("Errors:", errors)


Scenario OK: True
Errors: []


## 9. Flat trace export

`behavior_trace_to_records` merges transition, item, sequence, fork/join, decision, and merge steps for tooling. This cell rebuilds the same trace as section 8 so it runs **standalone** (you do not need prior cells in session memory).


In [9]:
_reset_aev_types()
cm = instantiate(AutonomousVehicle)
ctx = RunContext()
ctx.bind_input(cm.motion.auton_cmd_ok.stable_id, 1.0)
ctx.bind_input(cm.motion.min_supervisor_score.stable_id, 1.0)
ctx.bind_input(cm.vehicle_propulsion.powertrain.soc.stable_id, 0.9)
tr = BehaviorTrace()
dispatch_event(ctx, cm.vehicle_propulsion.powertrain, "enable_traction", trace=tr)
dispatch_event(ctx, cm.motion, "engage_auto", trace=tr)
dispatch_sequence(ctx, cm.safety, "preflight", trace=tr)
dispatch_fork_join(ctx, cm.safety, "dual_check", trace=tr)
emit_item(ctx, cm, cm.autonomy.perception.scene_out, "Observation", 20250324.0, trace=tr)

records = behavior_trace_to_records(tr)
for rec in records[:8]:
    print(rec)
print("...", len(records), "records total")


{'kind': 'transition', 'step_index': 0, 'part_path': 'AutonomousVehicle.vehicle_propulsion.powertrain', 'event_name': 'enable_traction', 'from_state': 'standby', 'to_state': 'propulsion', 'effect_name': 'apply_torque'}
{'kind': 'transition', 'step_index': 1, 'part_path': 'AutonomousVehicle.motion', 'event_name': 'engage_auto', 'from_state': 'manual', 'to_state': 'autonomous', 'effect_name': None}
{'kind': 'sequence', 'step_index': 2, 'part_path': 'AutonomousVehicle.safety', 'sequence_name': 'preflight'}
{'kind': 'fork_join', 'step_index': 3, 'part_path': 'AutonomousVehicle.safety', 'block_name': 'dual_check'}
{'kind': 'item_flow', 'step_index': 4, 'source_port_path': 'AutonomousVehicle.autonomy.perception.scene_out', 'target_port_path': 'AutonomousVehicle.autonomy.planner.fusion_in', 'item_kind': 'Observation', 'payload': 20250324.0}
{'kind': 'transition', 'step_index': 5, 'part_path': 'AutonomousVehicle.autonomy.planner', 'event_name': 'Observation', 'from_state': 'idle', 'to_state': 